In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

small_llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
from langchain_core.tools import tool


# tool 구성에는 세가지 구성이 필요하다
# 1. decorater (@tool)
# 2. description ("해당 툴의 설명")
# 3. expected arg ("인자")
@tool
def add(a: int, b: int) -> int:
    """숫자 a와 b를 더합니다."""
    return a+b

@tool
def multiply(a: int, b: int) -> int:
    """숫자 a와 b를 곱합니다."""
    return a*b


In [ ]:
%pip install -U duckduckgo-search
%pip install -qU langchain-google-community\[gmail\]

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [6]:
from langchain_google_community import GmailToolkit
from langchain_google_community.gmail.utils import (
    build_resource_service,
    get_gmail_credentials,
)

# Can review scopes here https://developers.google.com/gmail/api/auth/scopes
# For instance, readonly scope is 'https://www.googleapis.com/auth/gmail.readonly'
credentials = get_gmail_credentials(
    token_file="./google/token.json",
    scopes=["https://mail.google.com/"],
    client_secrets_file="./google/credentials.json",
)
api_resource = build_resource_service(credentials=credentials)
gmail_toolkit = GmailToolkit(api_resource=api_resource)
gamil_tool_list = gmail_toolkit.get_tools()

In [ ]:
%pip install arxiv
%pip install pymupdf

In [20]:
from langchain.agents import load_tools

loaded_tool_list = load_tools(["arxiv"])

In [1]:
import os

from langchain_chroma import Chroma
from langchain_core.tools.retriever import create_retriever_tool
from langchain_openai import OpenAIEmbeddings

embedding_function = OpenAIEmbeddings(model='text-embedding-3-large')
vector_store = Chroma(
    embedding_function=embedding_function,
    collection_name='real_estate_tax',
    persist_directory='./real_estate_tax_collection'
)
retriever = vector_store.as_retriever(search_kwargs={'k': 3})
retriever_tool = create_retriever_tool(
    retriever=retriever,
    name='real_estate_tax_retriever',
    description='Contains information about real estate tax up to December 2024',
)

In [7]:
from langgraph.prebuilt import ToolNode

tool_list = [add, multiply, search_tool, retriever_tool] + gamil_tool_list + loaded_tool_list
llm_with_tools = small_llm.bind_tools(tool_list)
tool_node = ToolNode(tools=tool_list)


In [8]:
from langgraph.graph import MessagesState, StateGraph
# 내장 state 사용
graph_builder = StateGraph(MessagesState)

In [9]:
# HumanMessage -> AIMessage -> ToolMessage ----> Messages 모두 추가
# 노드 선언

# llm_with_tools에 invoke만 하는 agent 노드
def agent(state: MessagesState) -> MessagesState:
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


In [10]:
# agent <-> tool_node 간 답변을 얻을떄 까지 메시지를 계속 교환 후 답변이 완료되면 end로 가야함
# 정보가 더 필요한지, 답변이 완료되었는지를 판단하기 위한 conditional 이 필요함

# tool_condition 에서 아래 코드가 정의되어있어 사용을 안해도 된다.
# def should_continue(state: MessagesState):
#     messages = state["messages"]
#     messages[-1].tool_calls
#     # 마지막 메시지가 tool_calls가 있는지 확인
#     # 있으면 True, 없으면 False
#     if len(messages[-1].tool_calls) > 0:
#         return 'tools'
#     return 'end'

In [ ]:
# 노드 추가
graph_builder.add_node("agent", agent)
graph_builder.add_node("tools", tool_node)

# shoud_continue는 conditional edge에서 분간하기 떄문에 노드로는 추가하지 않음 

In [ ]:
# 그래프 구성
from langgraph.graph import START, END

graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        'tools': "tools",
        'end': END
    }
)
graph_builder.add_edge("tools", "agent")



In [13]:
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import HumanMessage

# stream 모드로 실행해 agent가 어떤식으로 동작하는지 확인
for chunk in graph.stream({'messages': [HumanMessage('오늘의 날씨를 제 메일로 보내주세요 dkskrkwrr@gmail.com 입니다.')]}, stream_mode='values'):
    chunk['messages'][-1].pretty_print()
